In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import os
DATA_PATH = "/kaggle/input/modeling-wine/winequality-white new.csv"

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nMissing values total:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
df.head()

In [ ]:
# Parse price from the currency-like `alcohol` column
raw_price = df["alcohol"].astype(str)

df["price"] = pd.to_numeric(
    raw_price
    .str.replace("R$", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# Mixed formatting guard: allow at most one thousands-separator dot in raw strings
price_format_ok = raw_price.str.count(r"\.") <= 1

print("Price parse success:", int(df["price"].notna().sum()), "/", len(df))
print("Rows with suspicious raw price format:", int((~price_format_ok).sum()))
print("\nRaw price examples:")
print(raw_price.head(10).to_string(index=False))

In [ ]:
# Clean mixed-scale numeric features
# Heuristic repair for values likely shifted by decimal/thousands separators.
va = df["volatile acidity"].astype(float)
cl = df["chlorides"].astype(float)
den = df["density"].astype(float)

df["volatile acidity"] = np.where(va >= 10, va / 1000, np.where(va > 2, va / 10, va))
df["chlorides"] = np.where(cl >= 10, cl / 1000, np.where(cl > 1, cl / 100, cl))
df["density"] = np.where(
    den >= 100, den / 1000,
    np.where(den >= 10, den / 100, np.where(den > 2, den / 10, den))
)

feature_cols = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
    "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "quality"
]

work = df[feature_cols + ["price"]].copy()

# Keep rows with parseable, consistent prices and plausible chemistry ranges
mask = price_format_ok.copy()
for col, lo, hi in [
    ("volatile acidity", 0.05, 2.0),
    ("chlorides", 0.005, 1.0),
    ("density", 0.98, 1.20),
    ("pH", 2.5, 4.5),
]:
    mask &= work[col].between(lo, hi)

# Stable target regime in this file (removes likely contamination like 9, 10, 11)
mask &= work["price"].between(45000, 45600)

model_df = work.loc[mask].dropna().drop_duplicates().copy()

print("Rows after cleaning:", len(model_df))
print("\nPrice summary:\n", model_df["price"].describe())
model_df.head()


In [ ]:
# Quick EDA for modeling subset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(model_df["price"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Price Distribution")

corr = model_df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlation Heatmap")

plt.tight_layout()
plt.show()

print("Top correlations with price:")
print(corr["price"].drop("price").sort_values(key=lambda s: s.abs(), ascending=False).head(10))


In [ ]:
# Train/test split
X = model_df[feature_cols]
y = model_df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
# Baseline + boosted tree models
models = {
    "Baseline(mean)": DummyRegressor(strategy="mean"),
    "LightGBM": LGBMRegressor(
        n_estimators=700,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
    ),
}

results = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)

    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    fitted[name] = model

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


In [ ]:
# Feature importances for tree models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(["LightGBM", "XGBoost"]):
    model = fitted[name]
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    sns.barplot(x=imp.values, y=imp.index, ax=axes[i], color="teal")
    axes[i].set_title(f"{name} Feature Importance")
    axes[i].set_xlabel("Importance")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()
